# Controlled Descent Simulator — Quadrotor Model Derivation

**Author:** Diego Perazzolo, 2026
**License:** MIT

Companion to `dynamics_rocket_FFLQR01.ipynb`. The rocket and the quadrotor are two
*physical proxies* for the same controlled-descent problem. **What we port from the
rocket is only the trajectory** (`Poly4`, the flat-output reference). **The physics is
new**: a multirotor produces lateral acceleration only by *tilting its attitude* — there
is no direct side thrust — so the plant, the allocation, the feedforward and the LQR are
all redesigned here.

Pipeline (built **in order**):

1. **6-DOF quaternion model** — Newton–Euler, attitude as a unit quaternion, full Euler
   equation (gyroscopic term kept), rotor gyroscopic precession, linear drag. *(this notebook)*
2. **Control-allocation matrix** — the ArduPilot *QuadX* geometry mapping the four rotor
   thrusts to the virtual inputs `[F, τx, τy, τz]`. *(this notebook)*
3. **Feedforward via differential flatness** — flat outputs `σ = [x, y, z, ψ]`. *(next)*
4. **LQR** on the hover-linearized, minimally-parameterized error dynamics. *(next)*

### Modelling decisions frozen for this notebook
| topic | choice | why |
|---|---|---|
| world frame | **Z-up** (as in the rocket nb), gravity `[0,0,−mg]` | consistency with the rocket derivation |
| body frame | **FLU** (x fwd, y left, z up); thrust along `+z_b` | right-handed, matches Z-up world |
| motor layout | **ArduPilot QuadX** numbering + spin directions | natural bridge to ArduPilot SITL later |
| attitude | **unit quaternion** (Hamilton, scalar-first), body→world | no gimbal lock in aggressive near-vertical descents |
| translational drag | **linear**, world frame: `F_drag = −D·v` | enters the LQR `A` matrix as a constant (see §4 later) |
| Euler equation | full `I·ω̇ + ω×(I·ω) = τ` — **gyroscopic term kept** | `Izz` sizeable + aggressive manoeuvres ⇒ not negligible |
| rotor gyro | **kept**: `−ω × h_rotor` | roll↔pitch coupling under yaw with spinning props |

> Euler angles appear **only** for visualization; they never touch the dynamics.


In [ ]:
import sympy as sp
from sympy import symbols, Function, Matrix, eye, zeros, sqrt, sin, cos, Rational, diag, diff, lambdify
sp.init_printing(use_latex='mathjax', wrap_line=False)

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import control as ct

print("SymPy:", sp.__version__)
print("NumPy:", np.__version__)
print("control:", ct.__version__)

## 1. Symbolic 6-DOF model

The quadrotor is a rigid body of mass `m` with a diagonal inertia tensor
`diag(Ixx, Iyy, Izz)` (body axes aligned with the principal axes; for a symmetric quad
`Ixx ≈ Iyy` and `Izz` is the largest).

**State (13):** position `r` and velocity `v` in the **world** frame, attitude as the
unit quaternion `q` (body→world), and body rates `ω` in the **body** frame.

$$\mathbf{x} = [\,\underbrace{r_x,r_y,r_z}_{\text{world}},\ \underbrace{q_w,q_x,q_y,q_z}_{\text{body}\to\text{world}},\ \underbrace{v_x,v_y,v_z}_{\text{world}},\ \underbrace{\omega_x,\omega_y,\omega_z}_{\text{body}}\,]$$

**Virtual inputs (4):** collective thrust `F` (along `+z_b`) and body torques
`τ = [τx, τy, τz]`. The map from the four physical rotor thrusts `Tᵢ` to `[F, τ]` is the
allocation matrix of §2; here the dynamics are written directly in `[F, τ]`.

In [ ]:
# ---- parameters (symbolic) ----
t = symbols('t', real=True)
m, Ixx, Iyy, Izz, g = symbols('m I_xx I_yy I_zz g', positive=True)
d_x, d_y, d_z       = symbols('d_x d_y d_z', positive=True)      # world linear-drag diag
k_T, k_Q, I_rot     = symbols('k_T k_Q I_rot', positive=True)    # thrust/torque coeffs, rotor inertia
L                   = symbols('L', positive=True)               # arm length (centre -> motor)

# ---- state variables ----
r_x, r_y, r_z = symbols('r_x r_y r_z', real=True)
q_w, q_x, q_y, q_z = symbols('q_w q_x q_y q_z', real=True)
v_x, v_y, v_z = symbols('v_x v_y v_z', real=True)
w_x, w_y, w_z = symbols('w_x w_y w_z', real=True)      # body rates ω

# ---- virtual control inputs ----
F, tau_x, tau_y, tau_z = symbols('F tau_x tau_y tau_z', real=True)

q_vec = Matrix([q_w, q_x, q_y, q_z])
w_vec = Matrix([w_x, w_y, w_z])
v_vec = Matrix([v_x, v_y, v_z])
Imat  = diag(Ixx, Iyy, Izz)

### 1.1 Rotation matrix from the quaternion

With the **Hamilton, scalar-first** convention `q = [q_w, q_x, q_y, q_z]`, the body→world
rotation is

$$R(q) = \begin{bmatrix}
1-2(q_y^2+q_z^2) & 2(q_xq_y-q_wq_z) & 2(q_xq_z+q_wq_y)\\
2(q_xq_y+q_wq_z) & 1-2(q_x^2+q_z^2) & 2(q_yq_z-q_wq_x)\\
2(q_xq_z-q_wq_y) & 2(q_yq_z+q_wq_x) & 1-2(q_x^2+q_y^2)
\end{bmatrix}$$

so that `v_world = R(q)·v_body`. This replaces the rocket's `Rm = Rf1·Rf2·Rf3`.

In [ ]:
def R_of_q(qw, qx, qy, qz):
    return Matrix([
        [1-2*(qy**2+qz**2), 2*(qx*qy-qw*qz),   2*(qx*qz+qw*qy)],
        [2*(qx*qy+qw*qz),   1-2*(qx**2+qz**2), 2*(qy*qz-qw*qx)],
        [2*(qx*qz-qw*qy),   2*(qy*qz+qw*qx),   1-2*(qx**2+qy**2)],
    ])

R = R_of_q(q_w, q_x, q_y, q_z)
R

### 1.2 Attitude kinematics — the quaternion derivative

The attitude propagates as

$$\dot q = \tfrac12\, q \otimes \begin{bmatrix}0\\ \omega_{\text{body}}\end{bmatrix}
        = \tfrac12\, \Omega(\omega)\, q,\qquad
\Omega(\omega)=\begin{bmatrix}
0 & -\omega_x & -\omega_y & -\omega_z\\
\omega_x & 0 & \omega_z & -\omega_y\\
\omega_y & -\omega_z & 0 & \omega_x\\
\omega_z & \omega_y & -\omega_x & 0\end{bmatrix}.$$

Three things worth internalising, because they are exactly where Euler angles bite and
quaternions don't:

- **`ω` is not rotated.** It enters the formula in body coordinates directly — which is
  where the gyroscope measures it and where the Euler equation (§1.4) lives. There is no
  intermediate frame change.
- **There is no `W⁻¹` matrix.** With Euler angles you need `[φ̇,θ̇,ψ̇] = W⁻¹(φ,θ)·ω`, and
  `W⁻¹` blows up at pitch `±90°` (gimbal lock). The quaternion form has no such matrix and
  no singularity — the reason it's used for near-vertical descents.
- **Unit-norm drift.** `‖q‖=1` is only preserved exactly in continuous time; a numerical
  integrator drifts off the sphere, so we renormalize `q ← q/‖q‖` after each step.

Euler angles are recovered from `q` **only for plotting**, never for the dynamics.

In [ ]:
Omega = Matrix([
    [0,   -w_x, -w_y, -w_z],
    [w_x,  0,    w_z, -w_y],
    [w_y, -w_z,  0,    w_x],
    [w_z,  w_y, -w_x,  0  ],
])
q_dot = Rational(1, 2) * Omega * q_vec
q_dot

### 1.3 Translational dynamics (world frame)

$$m\,\dot v = R(q)\begin{bmatrix}0\\0\\F\end{bmatrix}
            + \begin{bmatrix}0\\0\\-mg\end{bmatrix}
            \;\underbrace{-\,R\,D\,R^{\!\top} v}_{\text{drag}},\qquad D=\mathrm{diag}(d_x,d_y,d_z).$$

All the thrust is along `+z_b`; it is **`R(q)` that tilts it** to produce horizontal
acceleration — there is no direct lateral thrust. Gravity is expressed in the world frame
(not rotated).

Drag is **linear and body-attached**: `D` is defined in the body frame and carried into the
world by `R·(−D·v_body) = −R D Rᵀ v`, so the drag ellipsoid **rotates with the airframe**
(an anisotropic `D` fixed to the world would keep more drag along world-`z` while the vehicle
tilts — physically wrong). Linear (not quadratic) is what makes this rotation clean and keeps
the term constant in the LQR `A` matrix: at hover `R=I`, so it linearizes to the constant
`−D`. Fidelity / gain-scheduling trade-offs of linear vs quadratic drag are noted in §4.

In [ ]:
D_mat        = diag(d_x, d_y, d_z)          # drag coeffs in the BODY frame
thrust_world = R * Matrix([0, 0, F])
gravity      = Matrix([0, 0, -m*g])
v_body       = R.T * v_vec                     # world velocity expressed in body
drag_world   = R * (-D_mat * v_body)           # body-attached: -R D Rᵀ v (rotates with craft)

v_dot = (thrust_world + gravity + drag_world) / m
v_dot

### 1.4 Rotational dynamics (body frame) — full Euler + rotor gyro

$$I\,\dot\omega + \omega\times(I\,\omega) = \tau \;-\; \omega \times h_{\text{rotor}}.$$

- **`ω×(I·ω)` is kept.** In the rocket it was dropped (axial symmetry + small `ω`); here
  `Izz` is substantial and manoeuvres are aggressive, so the gyroscopic coupling matters.
- **Rotor gyroscopic precession `−ω × h_rotor`.** The spinning props carry angular
  momentum `h_rotor = I_rot·(Σ σᵢ Ωᵢ)·ẑ_b` along the body-z axis, where `Ωᵢ = √(Tᵢ/k_T)`
  is the spin rate of rotor `i` and `σᵢ=±1` its direction. When the airframe rotates while
  the props spin, this couples roll↔pitch. The signed sum `S = Σ σᵢ Ωᵢ` is the *residual*
  rotor momentum — it is ~0 in balanced hover and grows during yaw/asymmetric commands.

We carry `S` as a symbol here; §2 shows it is built from the rotor thrusts through the
allocation. Note (important for §4): at hover `S=0` **and** `ω=0`, so both `ω×(Iω)` and the
rotor-gyro term drop out of the hover linearization — they live purely in the nonlinear
simulator.

In [ ]:
S_spin = symbols('S_spin', real=True)          # residual rotor momentum  Σ σ_i Ω_i
h_rotor = Matrix([0, 0, I_rot * S_spin])       # along body z
tau_vec = Matrix([tau_x, tau_y, tau_z])

w_dot = Imat.inv() * (tau_vec - w_vec.cross(Imat * w_vec) - w_vec.cross(h_rotor))
w_dot

### 1.5 Assembled state RHS `ẋ = f(x, u)`

In [ ]:
state = Matrix([r_x, r_y, r_z,
                q_w, q_x, q_y, q_z,
                v_x, v_y, v_z,
                w_x, w_y, w_z])

state_rhs = Matrix([
    v_x, v_y, v_z,          # ṙ = v
    *q_dot,                 # q̇ = ½ Ω(ω) q
    *v_dot,                 # v̇  (translational)
    *w_dot,                 # ω̇  (rotational)
])
print("state dim:", state.shape[0], " |  rhs dim:", state_rhs.shape[0])
state_rhs

### 1.6 Sanity check — dynamics vanish at hover

The single test the *model* needs: at hover (`q = identity`, `v = 0`, `ω = 0`, `F = mg`,
`τ = 0`, `S = 0`) every derivative must be zero.

In [ ]:
# numeric parameters for the ~2.4 kg / 550 mm airframe
params = {
    m: 2.4,
    Ixx: 0.025, Iyy: 0.025, Izz: 0.045,     # kg·m²  (estimates; Ixx≈Iyy, Izz larger)
    g:  9.81,
    d_x: 0.20, d_y: 0.20, d_z: 0.30,        # linear drag [N·s/m]
    k_T: 1.0e-5, k_Q: 1.6e-7,               # T = k_T ω² ,  Q = k_Q ω² ;  k_Q/k_T = 0.016 m
    I_rot: 3.0e-5,                          # rotor+prop inertia about spin axis
    L: 0.275,                               # arm length (wheelbase 550 mm ⇒ L = 0.275 m)
}
m_val, g_val = params[m], params[g]

f = sp.lambdify(
    (r_x, r_y, r_z, q_w, q_x, q_y, q_z, v_x, v_y, v_z,
     w_x, w_y, w_z, F, tau_x, tau_y, tau_z, S_spin),
    state_rhs.subs(params), 'numpy')

hover = f(0,0,0, 1,0,0,0, 0,0,0, 0,0,0, m_val*g_val, 0,0,0, 0.0)
hover = np.asarray(hover, float).flatten()
print("ẋ at hover:", np.round(hover, 12))
assert np.allclose(hover, 0, atol=1e-9), "hover is not an equilibrium!"
print("OK — hover is an equilibrium.")

### 1.7 Euler angles from the quaternion (visualization only)

Provided for plotting/telemetry. **Not used anywhere in the dynamics.**

In [ ]:
def euler_zyx_from_q(qw, qx, qy, qz):
    # (roll φ, pitch θ, yaw ψ) in rad from a scalar-first quaternion
    # standard aerospace Z-Y-X sequence — VISUALIZATION ONLY
    phi   = np.arctan2(2*(qw*qx + qy*qz), 1 - 2*(qx*qx + qy*qy))
    theta = np.arcsin(np.clip(2*(qw*qy - qz*qx), -1.0, 1.0))
    psi   = np.arctan2(2*(qw*qz + qx*qy), 1 - 2*(qy*qy + qz*qz))
    return phi, theta, psi

_phi, _th, _psi = euler_zyx_from_q(1, 0, 0, 0)
print(f"identity q -> (roll, pitch, yaw) = ({float(_phi):.3f}, {float(_th):.3f}, {float(_psi):.3f}) rad")

## 2. Control-allocation matrix — ArduPilot QuadX

The four rotor thrusts `Tᵢ = k_T ωᵢ²` map to the virtual inputs by

$$\begin{bmatrix}F\\ \tau_x\\ \tau_y\\ \tau_z\end{bmatrix}
= \underbrace{\begin{bmatrix}
1 & 1 & 1 & 1\\
y_1 & y_2 & y_3 & y_4\\
-x_1 & -x_2 & -x_3 & -x_4\\
-\sigma_1 c & -\sigma_2 c & -\sigma_3 c & -\sigma_4 c
\end{bmatrix}}_{\textstyle M}
\begin{bmatrix}T_1\\ T_2\\ T_3\\ T_4\end{bmatrix},
\qquad c=\frac{k_Q}{k_T}.$$

- `F  = Σ Tᵢ` (collective),
- `τx = Σ Tᵢ yᵢ` (roll, from motor y-position),
- `τy = −Σ Tᵢ xᵢ` (pitch, from motor x-position),
- `τz = Σ −σᵢ c Tᵢ` (yaw, via the drag-reaction torque — thrust and drag-torque are
  *siblings*, both `∝ ωᵢ²`, so `τ_drag,i = c·Tᵢ`).

**ArduPilot QuadX** numbering, top view, body **FLU** (`+x` fwd, `+y` left); with a 45° X
the offset is `d = L/√2`:

| motor | position | spin | σ |
|---|---|---|---|
| M1 front-right | `(+d, −d)` | CCW | +1 |
| M2 rear-left   | `(−d, +d)` | CCW | +1 |
| M3 front-left  | `(+d, +d)` | CW  | −1 |
| M4 rear-right  | `(−d, −d)` | CW  | −1 |

Diagonals share a spin direction, so hover yaw balances (`Σσᵢ = 0`).

In [ ]:
d   = L / sqrt(2)          # motor x,y offset on the 45° X
c_r = k_Q / k_T            # torque-to-thrust ratio

#              M1(FR)  M2(RL)  M3(FL)  M4(RR)
pos = [( d, -d), (-d,  d), ( d,  d), (-d, -d)]
sig = [   +1,      +1,      -1,      -1    ]      # +1 = CCW, -1 = CW

M_alloc = Matrix([
    [1, 1, 1, 1],
    [ pos[0][1],  pos[1][1],  pos[2][1],  pos[3][1]],      # τx = Σ T y
    [-pos[0][0], -pos[1][0], -pos[2][0], -pos[3][0]],      # τy = -Σ T x
    [-sig[0]*c_r, -sig[1]*c_r, -sig[2]*c_r, -sig[3]*c_r],  # τz = Σ -σ c T
])
print("det(M) =", sp.simplify(M_alloc.det()), " (≠ 0 ⇒ invertible)")
M_alloc

### 2.1 Numeric allocation + checks

Two checks the allocation needs: it inverts (so we can go `[F,τ] → Tᵢ`), and hover
(`F=mg, τ=0`) distributes as `mg/4` per rotor with zero residual rotor momentum.

In [ ]:
M_num  = np.array(M_alloc.subs(params)).astype(float)
M_inv  = np.linalg.inv(M_num)
print("cond(M) =", round(np.linalg.cond(M_num), 3))

# hover: F = mg, τ = 0
u_hover = np.array([m_val*g_val, 0, 0, 0])
T_hover = M_inv @ u_hover
print("hover rotor thrusts [N]:", np.round(T_hover, 4), "  (mg/4 =", round(m_val*g_val/4, 4), ")")

Omega_i = np.sqrt(np.clip(T_hover, 0, None) / params[k_T])
S_hover = float(sum(s*o for s, o in zip(sig, Omega_i)))
print("residual rotor momentum S at hover:", f"{S_hover:.3e}", " (≈ 0 ✓)")

def alloc_to_rotors(u):        # [F, τx, τy, τz] -> [T1..T4]
    return M_inv @ np.asarray(u, float)

def rotors_to_S(T):           # signed spin sum from rotor thrusts
    Om = np.sqrt(np.clip(T, 0, None) / params[k_T])
    return float(sum(s*o for s, o in zip(sig, Om)))

### 2.2 Motor layout — visual confirmation of numbering & spin

Worth *seeing* before we commit to it, since every sign in `M` follows from this picture.

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 5.2))
Ln = params[L]; dn = Ln/np.sqrt(2)
P = [( dn,-dn),(-dn, dn),( dn, dn),(-dn,-dn)]
names = ['M1 FR','M2 RL','M3 FL','M4 RR']
for (xx,yy),nm,s in zip(P, names, sig):
    # NOTE: plot uses screen axes = (y_left, x_fwd) so 'up' on screen is forward
    col = 'tab:blue' if s>0 else 'tab:red'
    ax.plot([0,yy],[0,xx],'k-',lw=2,zorder=1)
    ax.scatter(yy,xx,s=900,facecolor=col,edgecolor='k',zorder=2,alpha=.85)
    ax.text(yy,xx,nm.split()[0],ha='center',va='center',color='w',fontsize=9,zorder=3)
    spin = 'CCW' if s>0 else 'CW'
    ax.text(yy*1.35,xx*1.35,f"{nm.split()[1]}\n{spin}",ha='center',va='center',fontsize=8)
ax.annotate('',xy=(0,dn*1.9),xytext=(0,0),arrowprops=dict(arrowstyle='->',color='green',lw=2))
ax.text(0,dn*2.0,'+x  (forward)',ha='center',color='green')
ax.annotate('',xy=(dn*1.9,0),xytext=(0,0),arrowprops=dict(arrowstyle='->',color='purple',lw=2))
ax.text(dn*2.05,0,'+y (left)',va='center',color='purple')
ax.set_aspect('equal'); ax.set_xlim(-.55,.55); ax.set_ylim(-.55,.55)
ax.set_title('ArduPilot QuadX — FLU top view\n(blue = CCW σ=+1, red = CW σ=−1)')
ax.grid(alpha=.3); ax.invert_xaxis()   # +y left points left on screen
plt.tight_layout(); plt.show()

## 3. Feedforward via differential flatness

The quadrotor is **differentially flat** with flat outputs `σ = [x, y, z, ψ]` (CoM position +
heading). Flatness means every state and input is an **algebraic** function of `σ` and a finite
number of its time-derivatives — no ODE to integrate. That is exactly what makes an *exact*
feedforward possible, and it lines up 1:1 with the `Reference_t` struct (`pos/vel/acc/jerk/snap`):

| flat-output derivative | yields | FF channel |
|---|---|---|
| `acc` (2ⁿᵈ) | thrust `F` + attitude (2 of 3 DOF), `ψ` fixes the 3ʳᵈ | §3.1 |
| `jerk` (3ʳᵈ) | body rates `ω_ref` | §3.2 |
| `snap` (4ᵗʰ) | angular acceleration → torques `τ_ff` | §3.3 |

The design rule below is **reuse, not re-derivation**: the flat map inverts relations we already
built in §1 (`thrust_world`, the Euler equation `w_dot`), it does not restate them.

### 3.1 Attitude & thrust from acceleration

Newton in world (drag ignored in the FF, as in the rocket), with `z_B = R·ẑ_b` the body-z axis
in world:

$$m\ddot r = F\,z_B - m g\,\hat z_w \;\Longrightarrow\; F\,z_B = m(\ddot r + g\hat z_w) \equiv m\,\mathbf a.$$

This is the same `thrust_world = R·[0,0,F]` relation from §1.3, read backwards. Split it into
magnitude and direction, then close the last DOF with the heading `ψ`:

$$F_{\text{ff}} = m\lVert\mathbf a\rVert,\quad z_B=\tfrac{\mathbf a}{\lVert\mathbf a\rVert},\quad
x_C=[\cos\psi,\sin\psi,0],\quad y_B=\tfrac{z_B\times x_C}{\lVert z_B\times x_C\rVert},\quad x_B=y_B\times z_B.$$

Well-defined exactly on the operating envelope: `‖a‖>0` fails only at pure free-fall
(`no F=0`), and `z_B ∥ x_C` fails only at pitch `±90°` — the two limits you named.

In [ ]:
# reference acc + heading (generic symbols, for the closed-form map)
axr, ayr, azr, psir = symbols('a_x a_y a_z psi', real=True)

def flat_thrust_attitude(acc, psi_ref):
    # invert the §1.3 relation  thrust_world = R·[0,0,F] = m·(acc + g·ẑ)
    a_th = Matrix([acc[0], acc[1], acc[2] + g])               # reuse of thrust_world
    n_a  = sqrt(a_th.dot(a_th))
    F_ff = m * n_a
    z_B  = a_th / n_a                                         # body-z in world
    x_C  = Matrix([cos(psi_ref), sin(psi_ref), 0])            # heading from flat output ψ
    y_B  = z_B.cross(x_C); y_B = y_B / sqrt(y_B.dot(y_B))
    x_B  = y_B.cross(z_B)
    return F_ff, Matrix.hstack(x_B, y_B, z_B)                 # R_ref = [x_B | y_B | z_B]

F_ff_generic, _ = flat_thrust_attitude([axr, ayr, azr], psir)
print("F_ff  ="); F_ff_generic

### 3.2 Body rates from jerk

No closed-form rate formula is typed in — the body rates fall straight out of the attitude via
the kinematic identity `[ω]_× = R_refᵀ·Ṙ_ref` (reusing `R_ref`). This is the quaternion-era
replacement for the rocket's `α̇_ff, β̇_ff`, and it fills the angular-velocity entries of the LQR
reference state so the controller doesn't read `ω_ref` as tracking error.

In [ ]:
def body_rates_from_attitude(R_ref):
    W = R_ref.T * diff(R_ref, t)                 # [ω]_×  = R_refᵀ Ṙ_ref  (reuse R_ref)
    return Matrix([W[2, 1], W[0, 2], W[1, 0]])   # vee(·)

### 3.3 Torques from snap

The torque FF is just the **§1.4 Euler equation inverted**: solve `w_dot = τ` for `τ` given the
reference `ω̇_ref = ω̇_ref(snap)`. We reuse the same `Imat`; the rotor-gyro term is dropped here
(a small, plant-only effect — the LQR compensates it, exactly as the rocket FF dropped drag):

$$\tau_{\text{ff}} = I\,\dot\omega_{\text{ref}} + \omega_{\text{ref}}\times(I\,\omega_{\text{ref}}).$$

In [ ]:
def ff_torque(omega_ref, Imat):
    omega_dot_ref = diff(omega_ref, t)                              # ω̇_ref from snap
    return Imat*omega_dot_ref + omega_ref.cross(Imat*omega_ref)     # invert §1.4 Euler (gyro dropped)

### 3.4 Reference trajectory (Poly4) and the assembled FF

Only the **trajectory** is ported from the rocket — the same degree-4 closed form from boundary
conditions `(p0, v0, pF, vF, aF, T)`. Here a gentle descent for the 2.4 kg airframe, heading held
(`ψ = 0`; a yaw slew is a one-line change). We substitute this trajectory into the flat map of
§3.1–3.3 to get `F_ff, R_ref, ω_ref, τ_ff` as concrete functions of time.

In [ ]:
# Poly4 per axis from boundary conditions (closed form, ported from the rocket)
def poly4(p0, v0, pF, vF, aF, T):
    a2 = ( T**2*aF - 6*T*v0 - 6*T*vF - 12*p0 + 12*pF) / (2*T**2)
    a3 = (-T**2*aF + 3*T*v0 + 5*T*vF +  8*p0 -  8*pF) / T**3
    a4 = ( T**2*aF - 2*T*v0 - 4*T*vF -  6*p0 +  6*pF) / (2*T**4)
    return p0 + v0*t + a2*t**2 + a3*t**3 + a4*t**4

TF = 8.0
Tj = Matrix([ poly4(-8.0,  0.0, 0.0, 0.0, 0.0, TF),    # x: -8 -> 0
              poly4( 8.0,  2.0, 0.0, 0.0, 0.0, TF),    # y:  8 -> 0
              poly4(30.0, -4.0, 0.0, 0.0, 0.0, TF) ])  # z: 30 -> 0 (descent)
psi_ref = sp.Integer(0)                                # heading held

acc_ref = diff(Tj, t, 2)

# assemble the FF along the trajectory (reusing the §3.1-3.3 helpers)
F_ff_t, R_ref_t = flat_thrust_attitude([acc_ref[0], acc_ref[1], acc_ref[2]], psi_ref)
omega_ref_t     = body_rates_from_attitude(R_ref_t)
tau_ff_t        = ff_torque(omega_ref_t, Imat)
print("FF assembled along the trajectory: F_ff_t, R_ref_t, omega_ref_t, tau_ff_t")

### 3.5 Acid test — is the feedforward exact?

Two checks, no controller yet:
1. **Pointwise consistency** — does `R_ref·[0,0,F_ff]/m − g·ẑ` reproduce `acc_ref` exactly?
2. **Open-loop** — feed only the FF into the nonlinear plant (drag & rotor-gyro **off**, the
   best case, as in the rocket's §7) from an on-reference start. Exact FF ⇒ the residual stays
   at numerical zero across the whole descent. Any wrong sign would diverge.

In [ ]:
from scipy.spatial.transform import Rotation
from scipy.integrate import solve_ivp
from scipy.interpolate import CubicSpline

# lambdify references (subs numeric params)
Tj_f    = lambdify(t, Tj.subs(params),          'numpy')
Vj_f    = lambdify(t, diff(Tj, t).subs(params), 'numpy')
acc_f   = lambdify(t, acc_ref.subs(params),     'numpy')
F_ff_f  = lambdify(t, F_ff_t.subs(params),      'numpy')
R_ref_f = lambdify(t, R_ref_t.subs(params),     'numpy')
om_ff_f = lambdify(t, omega_ref_t.subs(params), 'numpy')
tau_ff_f= lambdify(t, tau_ff_t.subs(params),    'numpy')

# (1) pointwise translational consistency
res = 0.0
for tt in np.linspace(0, TF, 60):
    Rr = np.array(R_ref_f(tt), float).reshape(3, 3)
    acc_from_ff = (Rr @ np.array([0, 0, float(F_ff_f(tt))]))/m_val + np.array([0, 0, -g_val])
    res = max(res, np.linalg.norm(acc_from_ff - np.array(acc_f(tt), float).flatten()))
print(f"(1) max translational-consistency residual: {res:.2e}")

# best-case plant (drag OFF, rotor-gyro OFF) — reuse state_rhs
state_rhs_bc = state_rhs.subs({d_x: 0, d_y: 0, d_z: 0}).subs(params)
f_bc = lambdify((r_x, r_y, r_z, q_w, q_x, q_y, q_z, v_x, v_y, v_z,
                 w_x, w_y, w_z, F, tau_x, tau_y, tau_z, S_spin), state_rhs_bc, 'numpy')

def q_from_R(Rm):
    qq = Rotation.from_matrix(Rm).as_quat()      # [x,y,z,w]
    return np.array([qq[3], qq[0], qq[1], qq[2]])

# precompute FF on a grid for a fast open-loop integration
tg = np.linspace(0, TF, 400)
Fs = CubicSpline(tg, np.array([float(F_ff_f(tt)) for tt in tg]))
Ts = CubicSpline(tg, np.array([np.array(tau_ff_f(tt), float).flatten() for tt in tg]), axis=0)

def plant_bc(tt, s):
    q = s[3:7]; q = q / np.linalg.norm(q)
    return np.array(f_bc(*s[:3], *q, *s[7:10], *s[10:13],
                         float(Fs(tt)), *Ts(tt), 0.0)).flatten()

R0 = np.array(R_ref_f(0.0), float).reshape(3, 3)
s0 = np.concatenate([np.array(Tj_f(0.0), float).flatten(), q_from_R(R0),
                     np.array(Vj_f(0.0), float).flatten(),
                     np.array(om_ff_f(0.0), float).flatten()])
sol = solve_ivp(plant_bc, (0, TF), s0, t_eval=np.linspace(0, TF, 200), rtol=1e-9, atol=1e-11)
ref_pos = np.array([np.array(Tj_f(tt), float).flatten() for tt in sol.t]).T
pos_err = np.linalg.norm(sol.y[:3] - ref_pos, axis=0)
print(f"(2) open-loop FF position error: max = {pos_err.max():.2e} m, final = {pos_err[-1]:.2e} m")
print(f"    landing at {np.round(sol.y[:3, -1], 4)}  (target [0, 0, 0])")

### 3.6 Feedforward profiles

Thrust, torque and the FF attitude along the descent. `F_ff` must stay within the rotor envelope
`[0, 4·T_max]` (it's the four props summed); the FF attitude is shown as Euler angles (viz only).

In [ ]:
t_grid = np.linspace(0, TF, 300)
F_hist   = np.array([float(F_ff_f(tt)) for tt in t_grid])
tau_hist = np.array([np.array(tau_ff_f(tt), float).flatten() for tt in t_grid])
ang_hist = np.array([np.degrees(euler_zyx_from_q(*q_from_R(np.array(R_ref_f(tt), float).reshape(3, 3))))
                     for tt in t_grid])
T_max_rotor = 36.0     # per-rotor max thrust [N] (Hobbywing XRotor 2812, ~3.7 kgf)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].plot(t_grid, F_hist, lw=1.6)
ax[0].axhline(m_val*g_val, color='g', ls=':', label='hover mg')
ax[0].axhline(4*T_max_rotor, color='r', ls='--', label='4·T_max')
ax[0].axhline(0, color='r', ls='--')
ax[0].set_ylabel('F_ff [N]'); ax[0].set_xlabel('t [s]'); ax[0].set_title('Thrust FF')
ax[0].legend(); ax[0].grid(alpha=.3)

for i, nm in enumerate(['τx (roll)', 'τy (pitch)', 'τz (yaw)']):
    ax[1].plot(t_grid, tau_hist[:, i], lw=1.4, label=nm)
ax[1].set_ylabel('τ_ff [N·m]'); ax[1].set_xlabel('t [s]'); ax[1].set_title('Torque FF')
ax[1].legend(); ax[1].grid(alpha=.3)

for i, nm in enumerate(['roll φ', 'pitch θ', 'yaw ψ']):
    ax[2].plot(t_grid, ang_hist[:, i], lw=1.4, label=nm)
ax[2].set_ylabel('attitude [deg]'); ax[2].set_xlabel('t [s]'); ax[2].set_title('FF attitude (viz)')
ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 4. LQR on the hover-linearized error dynamics

The LQR corrects the tracking error around the feedforward. Four steps: linearize the plant at
hover, reduce the quaternion to a minimal 3-vector attitude error `δθ`, augment with four error
integrators (`∫x, ∫y, ∫z, ∫ψ`), and solve the Riccati equation.

Why the yaw integrator (four, not three as in the rocket): `ψ` is an independent flat output we
actively track, so its steady-state error must be nulled like position's. The rocket held roll at
zero and needed no such integrator.

### 4.1 Jacobians and the hover equilibrium (derived)

`A = ∂f/∂x`, `B = ∂f/∂u` from the **13-state** `state_rhs` (reused). The equilibrium is *derived*,
not assumed: solve `f = 0` at the identity quaternion with zero velocities/rates for the inputs —
it returns `F = mg`, `τ = 0`.

In [ ]:
control_vec = Matrix([F, tau_x, tau_y, tau_z])
A13 = state_rhs.jacobian(state)          # reuse state_rhs / state
B13 = state_rhs.jacobian(control_vec)

# hover equilibrium (derived): f = 0 at q = identity, v = ω = 0, S = 0
hover_state = {q_w:1, q_x:0, q_y:0, q_z:0, v_x:0, v_y:0, v_z:0,
               w_x:0, w_y:0, w_z:0, r_x:0, r_y:0, r_z:0, S_spin:0}
hover_in = sp.solve(state_rhs.subs(hover_state), [F, tau_x, tau_y, tau_z], dict=True)[0]
print("hover inputs (derived):", {str(k): v for k, v in hover_in.items()})

eq_subs = {**hover_state, **hover_in, **params}
A13n = np.array(A13.subs(eq_subs), dtype=float)
B13n = np.array(B13.subs(eq_subs), dtype=float)
print("A13:", A13n.shape, " B13:", B13n.shape)

### 4.2 Reduce the quaternion to `δθ`

The 13-state Jacobian is singular along the quaternion norm direction (the unit constraint). We
project onto the 12-dim tangent space using the map `δq = G·δθ` with `G = ½[0; I₃]` (and its
left-inverse `G⁺ = [0, 2I₃]`). The minimal state is `[δr, δθ, δv, δω]`; the hallmark hover
coupling `δv̇ₓ = +g·δθ_y`, `δv̇_y = −g·δθ_x` should appear.

In [ ]:
G  = np.zeros((4, 3)); G[1:, :] = 0.5*np.eye(3)     # dq/dδθ  at identity
Gp = np.zeros((3, 4)); Gp[:, 1:] = 2.0*np.eye(3)     # left-inverse

# column map (minimal->full tangent) and row map (full->minimal), block by block
Tm = np.zeros((13, 12)); Pm = np.zeros((12, 13))
for (r0, c0, blk) in [(0,0,np.eye(3)), (3,3,G),  (7,6,np.eye(3)), (10,9,np.eye(3))]:
    Tm[r0:r0+blk.shape[0], c0:c0+blk.shape[1]] = blk
for (r0, c0, blk) in [(0,0,np.eye(3)), (3,3,Gp), (6,7,np.eye(3)), (9,10,np.eye(3))]:
    Pm[r0:r0+blk.shape[0], c0:c0+blk.shape[1]] = blk

A12 = Pm @ A13n @ Tm
B12 = Pm @ B13n
print(f"hover coupling check:  A12[δv̇ₓ, δθ_y] = {A12[6,4]:+.2f} (=+g),   "
      f"A12[δv̇_y, δθ_x] = {A12[7,3]:+.2f} (=-g)")

### 4.3 Augment with four error integrators

State order `[δr(3), δθ(3), δv(3), δω(3), ∫x, ∫y, ∫z, ∫ψ]` (16). The integrators accumulate the
*negative* errors (`d/dt[∫x] = −δr_x`, `d/dt[∫ψ] = −δθ_z`), which in the regulator is the LQR form
of `∫(ref − state)`.

In [ ]:
A_e = np.zeros((16, 16)); A_e[:12, :12] = A12
A_e[12, 0] = -1.0     # ∫x  <- -δr_x
A_e[13, 1] = -1.0     # ∫y  <- -δr_y
A_e[14, 2] = -1.0     # ∫z  <- -δr_z
A_e[15, 5] = -1.0     # ∫ψ  <- -δθ_z  (yaw)
B_e = np.zeros((16, 4)); B_e[:12, :] = B12
print("augmented:", A_e.shape, B_e.shape)

### 4.4 LQR weights and gain

Position and heading heavy (that is what we land on), rates penalized (no spinning), thrust cheap
relative to torque in `R`.

In [ ]:
Q = np.eye(16)
Q[0,0]=Q[1,1]=Q[2,2]   = 1200                 # position
Q[3,3]=Q[4,4]=120;  Q[5,5]=200                # roll, pitch, yaw
Q[6,6]=Q[7,7]=Q[8,8]   = 2                    # linear velocity
Q[9,9]=Q[10,10]=Q[11,11]=80                   # body rates
Q[12,12]=Q[13,13]=Q[14,14]=40; Q[15,15]=40    # integrators (x,y,z,ψ)
R_w = np.diag([1e-2, 4.0, 4.0, 4.0])          # thrust cheap, torques dearer

K, S, E = ct.lqr(A_e, B_e, Q, R_w)
print("K:", K.shape, " max Re(eig):", round(E.real.max(), 4))
assert np.all(E.real < 0), "closed loop unstable — retune Q, R"
print("closed loop stable.")

### 4.5 Closed loop — FF + LQR, drag & rotor-gyro ON, perturbed start

The realistic case: the full nonlinear plant (`f` from §1.6, drag and rotor-gyro active), the FF
of §3, and the LQR correction on the tracking error. The attitude error is the rotation vector of
`R_refᵀ·R` (multiplicative, no Euler subtraction). Control is saturated **at the rotor** — allocate
`u → Tᵢ`, clip each to `[0, T_max]`, recombine — the physically correct place.

In [ ]:
from scipy.spatial.transform import Rotation   # (also used in §3.5)
T_max_rotor = 36.0                                # per-rotor max thrust [N]

def R_num(q):
    return Rotation.from_quat([q[1], q[2], q[3], q[0]]).as_matrix()

def closed_loop(tt, S16):
    s, integ = S16[:13], S16[13:17]
    q = s[3:7] / np.linalg.norm(s[3:7])
    R_cur = R_num(q); R_r = np.array(R_ref_f(tt), float).reshape(3, 3)

    dr  = s[0:3]   - np.array(Tj_f(tt), float).flatten()
    dth = Rotation.from_matrix(R_r.T @ R_cur).as_rotvec()       # δθ, minimal attitude error
    dv  = s[7:10]  - np.array(Vj_f(tt), float).flatten()
    dw  = s[10:13] - np.array(om_ff_f(tt), float).flatten()
    err = np.concatenate([dr, dth, dv, dw, integ])

    u_ff = np.concatenate([[float(Fs(tt))], Ts(tt)])            # reuse §3.5 FF splines
    u    = u_ff - K @ err

    T_rotor = np.clip(M_inv @ u, 0.0, T_max_rotor)             # rotor-level saturation
    u_sat   = M_num @ T_rotor
    S_val   = float(sum(sg*np.sqrt(max(Ti, 0)/params[k_T]) for sg, Ti in zip(sig, T_rotor)))

    dx = np.array(f(*s[:3], *q, *s[7:10], *s[10:13],
                    u_sat[0], u_sat[1], u_sat[2], u_sat[3], S_val)).flatten()
    d_int = np.array([-dr[0], -dr[1], -dr[2], -dth[2]])
    return np.concatenate([dx, d_int])

# perturbed initial condition, off the reference
r0 = np.array(Tj_f(0.), float).flatten() + np.array([0.5, -0.5, 1.0])
v0 = np.array(Vj_f(0.), float).flatten() + np.array([0.2, 0.2, -0.3])
R0p = np.array(R_ref_f(0.), float).reshape(3, 3) @ Rotation.from_euler('xyz', [3, 3, 0], degrees=True).as_matrix()
S0 = np.concatenate([r0, q_from_R(R0p), v0, np.array(om_ff_f(0.), float).flatten(), [0, 0, 0, 0]])

sol_cl = solve_ivp(closed_loop, (0, TF), S0, t_eval=np.linspace(0, TF, 300), rtol=1e-8, atol=1e-10)
ref_cl = np.array([np.array(Tj_f(tt), float).flatten() for tt in sol_cl.t]).T
pe_cl  = np.linalg.norm(sol_cl.y[:3] - ref_cl, axis=0)
print(f"FF+LQR tracking:  init err = {pe_cl[0]:.3f} m,  max = {pe_cl.max():.3f} m,  final = {pe_cl[-1]:.4f} m")
print(f"landing at {np.round(sol_cl.y[:3, -1], 3)}  (target [0, 0, 0])")

### 4.6 Tracking plots

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

t_ref = np.linspace(0, TF, 200)
ref_traj = np.array([np.array(Tj_f(tt), float).flatten() for tt in t_ref]).T

fig = plt.figure(figsize=(13, 5))
axA = fig.add_subplot(1, 2, 1, projection='3d')
axA.plot(*ref_traj, 'r--', lw=2, alpha=.7, label='reference')
axA.plot(sol_cl.y[0], sol_cl.y[1], sol_cl.y[2], 'b-', lw=1.4, label='FF+LQR')
axA.scatter(*S0[:3], color='g', s=60, label='start (perturbed)')
axA.scatter(0, 0, 0, color='r', marker='x', s=60, label='target')
axA.set_xlabel('x'); axA.set_ylabel('y'); axA.set_zlabel('z'); axA.legend(); axA.set_title('3D trajectory')

axB = fig.add_subplot(1, 2, 2)
axB.plot(sol_cl.t, pe_cl, lw=1.5)
axB.set_xlabel('t [s]'); axB.set_ylabel('‖position error‖ [m]')
axB.set_title('Tracking error decay'); axB.grid(alpha=.3)
plt.tight_layout(); plt.show()

# control history with rotor-level saturation
U = np.array([np.clip(M_inv @ (np.concatenate([[float(Fs(tt))], Ts(tt)])
             - K @ np.concatenate([
                 sol_cl.y[0:3, i] - np.array(Tj_f(tt), float).flatten(),
                 Rotation.from_matrix(np.array(R_ref_f(tt), float).reshape(3,3).T @ R_num(sol_cl.y[3:7, i]/np.linalg.norm(sol_cl.y[3:7, i]))).as_rotvec(),
                 sol_cl.y[7:10, i] - np.array(Vj_f(tt), float).flatten(),
                 sol_cl.y[10:13, i] - np.array(om_ff_f(tt), float).flatten(),
                 sol_cl.y[13:17, i]])), 0.0, T_max_rotor)
             for i, tt in enumerate(sol_cl.t)])
fig, axc = plt.subplots(figsize=(9, 3.4))
for j in range(4):
    axc.plot(sol_cl.t, U[:, j], lw=1.2, label=f'T{j+1}')
axc.axhline(T_max_rotor, color='r', ls='--', alpha=.6, label='T_max')
axc.axhline(0, color='r', ls='--', alpha=.6)
axc.set_xlabel('t [s]'); axc.set_ylabel('rotor thrust [N]')
axc.set_title('Per-rotor commands (saturated)'); axc.legend(ncol=5); axc.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 5. Discussion and limitations

Deliberate trade-offs, documented honestly (portfolio value), not bugs.

**LQR is linearized at hover, not along the trajectory.** `K` is computed once at the hover
equilibrium; the descent leaves that neighbourhood, so `K` is stabilizing but suboptimal away from
hover. The textbook fix is TVLQR (integrate the differential Riccati equation backwards along the
reference); out of scope here.

**Linear drag and gain scheduling.** The FF ignores drag; the LQR uses the linearized (constant)
drag in `A`. A quadratic drag `−D|v|v` would linearize to `2D|v₀|` — a coefficient that *depends on
the descent speed `v₀`*, so a single `K` would no longer fit and we'd need **gain scheduling** (one
`K` per descent-speed band). Keeping drag linear is what buys a single, clean `K`. Revisit if
high-speed fidelity demands the quadratic model.

**Rotor-gyro is in the plant but not the FF.** A small, plant-only effect on this gentle descent;
the LQR mops up the residual. For aggressive descents it could be folded into the FF (don't zero
`S_spin` in `ff_torque`, iterating once on the allocation).

**Attitude error is multiplicative** (`R_refᵀ·R` → rotation vector), so it stays valid for large
attitude excursions — no small-angle Euler subtraction that would break near pitch `±90°`.

**Yaw-frame compensation (important).** The LQR gain is synthesized at `yaw = 0`, where body-x is
world-x and the hover coupling is `δv̇ₓ = g·δθ_y`. At a non-zero heading `ψ` the true horizontal
position→attitude→acceleration coupling is rotated by `ψ`, so feeding K the *world-frame* horizontal
position/velocity errors misdirects the feedback; the x/y loop is stable near `ψ=0` but its closed-
loop eigenvalue crosses into the right half-plane as `ψ` grows (marginal near `ψ≈0.5`, clearly
unstable by `ψ≈0.7`, Re≈+0.15). The fix, applied in `ExecuteControl`, is to express the horizontal
position, velocity and integral errors in the heading frame — rotate them by `Rz(−ψ)` — before
applying K; the attitude/rate errors are already in the body frame. With this, the closed loop is
stable at all yaw. (A fuller alternative would be gain-scheduling K on `ψ`, or TVLQR.)

---
Plant (§1) → allocation (§2) → exact flatness feedforward (§3) → LQR with four integrators (§4),
all cross-checked. What remains is the C++/WASM export (§6).


## 6. C++ code export for the runtime

The C++/WASM runtime consumes a generated `FF_LQR_01` class (a `.hpp/.cpp` pair). The generator
is a `QuadCodegen` that reuses the shared `BaseCodegen` machinery (header, accessors, param access,
`K_e` literal) and overrides only the two vehicle-specific emitters. Key differences from the
rocket generator, all handled **inside the generated code**:

- **Runtime state is 17** — `[r(3), q(4), v(3), ω(3), ∫x, ∫y, ∫z, ∫ψ]` — the integrator carries the
  quaternion; the **LQR error is 16** (`δθ` is a 3-vector), so `K_e` is `4×16` and `ExecuteControl`
  reduces `q → δθ` at runtime via the multiplicative error `δθ = rotvec(R_refᵀ·R(q))`.
- **The feedforward emits `R_ref` construction + `ω_ref` + `τ_ff`**, not two tilt angles.
- **`InputVec = [T₁..T₄]`** — `ExecuteControl` allocates `[F,τ] → Tᵢ` and saturates each rotor to
  `[0, T_max]` (like the rocket saturated its actuators); `Dynamics` takes `Tᵢ` and recovers
  `F/τ/S_spin` internally.

This section first puts the feedforward in the **static form** the generated code needs.

### 6.1 Feedforward in static form

`ExecuteControl` receives a `Reference_t` (`pos/vel/acc/jerk/snap`) and the heading `ψ` — not a
time parameter. So the FF must be expressed in those **independent** reference symbols. We get it
by feeding the §3 helpers **time-functions** of the flat outputs and then rewriting each time
derivative as an independent symbol: `d(acc)/dt → jerk`, `d²(acc)/dt² → snap`, `dψ/dt → ψ̇`, etc.
Nothing is re-derived — the same `flat_thrust_attitude`, `body_rates_from_attitude`, `ff_torque`
are reused.

In [ ]:
from sympy import Function

# feed the §3 helpers time-functions of the flat outputs
ax_t, ay_t, az_t = Function('a_x')(t), Function('a_y')(t), Function('a_z')(t)
psi_t = Function('psi')(t)

F_ff_tf, R_ref_tf = flat_thrust_attitude([ax_t, ay_t, az_t], psi_t)   # reuse §3.1
omega_ref_tf      = body_rates_from_attitude(R_ref_tf)                # reuse §3.2
tau_ff_tf         = ff_torque(omega_ref_tf, Imat)                     # reuse §3.3

# independent reference symbols (axr, ayr, azr, psir already defined in §3.1)
jxr, jyr, jzr = symbols('j_x j_y j_z', real=True)     # jerk
sxr, syr, szr = symbols('s_x s_y s_z', real=True)     # snap
psidr, psiddr = symbols('psi_dot psi_ddot', real=True)

def staticize(expr):
    # rewrite time-derivatives as independent symbols (highest order first)
    reps = [(diff(ax_t,t,2),sxr),(diff(ay_t,t,2),syr),(diff(az_t,t,2),szr),
            (diff(ax_t,t),jxr),  (diff(ay_t,t),jyr),  (diff(az_t,t),jzr),
            (ax_t,axr),(ay_t,ayr),(az_t,azr),
            (diff(psi_t,t,2),psiddr),(diff(psi_t,t),psidr),(psi_t,psir)]
    for a, b in reps:
        expr = expr.subs(a, b)
    return expr

F_ff_static     = staticize(F_ff_tf)
R_ref_static    = R_ref_tf.applyfunc(staticize)
omega_ff_static = omega_ref_tf.applyfunc(staticize)
tau_ff_static   = tau_ff_tf.applyfunc(staticize)

print("F_ff_static     depends on:", sorted(str(s) for s in F_ff_static.free_symbols))
print("omega_ff_static depends on:", sorted(str(s) for s in omega_ff_static.free_symbols))
print("tau_ff_static   depends on:", sorted(str(s) for s in tau_ff_static.free_symbols))

### 6.2 Validate the static form against §3

The static form must reproduce the §3 trajectory feedforward exactly when the specific
`(acc, jerk, snap, ψ, ψ̇, ψ̈)` of the Poly4 descent are substituted back in.

In [ ]:
traj_subs = {axr: acc_ref[0], ayr: acc_ref[1], azr: acc_ref[2],
             jxr: diff(Tj[0],t,3), jyr: diff(Tj[1],t,3), jzr: diff(Tj[2],t,3),
             sxr: diff(Tj[0],t,4), syr: diff(Tj[1],t,4), szr: diff(Tj[2],t,4),
             psir: 0, psidr: 0, psiddr: 0}

fF_s   = lambdify(t, F_ff_static.subs(traj_subs).subs(params),     'numpy')
fR_s   = lambdify(t, R_ref_static.subs(traj_subs).subs(params),    'numpy')
fom_s  = lambdify(t, omega_ff_static.subs(traj_subs).subs(params), 'numpy')
ftau_s = lambdify(t, tau_ff_static.subs(traj_subs).subs(params),   'numpy')

eF = eR = eom = etau = 0.0
for tt in np.linspace(0.2, TF-0.2, 40):
    eF   = max(eF,   abs(float(fF_s(tt)) - float(F_ff_f(tt))))
    eR   = max(eR,   np.max(np.abs(np.array(fR_s(tt), float) - np.array(R_ref_f(tt), float))))
    eom  = max(eom,  np.max(np.abs(np.array(fom_s(tt), float).flatten() - np.array(om_ff_f(tt), float).flatten())))
    etau = max(etau, np.max(np.abs(np.array(ftau_s(tt), float).flatten() - np.array(tau_ff_f(tt), float).flatten())))
print(f"static vs §3   F: {eF:.2e}   R_ref: {eR:.2e}   omega: {eom:.2e}   tau: {etau:.2e}")
assert max(eF, eR, eom, etau) < 1e-9, "static FF disagrees with §3"
print("static-form feedforward matches §3 to machine precision.")

### 6.3 Runtime model in `Tᵢ` form, and emission

The runtime state is driven by the four rotor thrusts. We reuse the §2 allocation to recover
`[F, τ]` and the residual rotor momentum `S` from `[T₁..T₄]`, substitute those into the §1
`state_rhs` (renaming the drag to the struct's `c/cz`), and append the four integrators. Then
`QuadCodegen` (built on the shared `BaseCodegen`) emits the `.hpp/.cpp`.

In [ ]:
import quad_codegen as qcg
from importlib import reload; reload(qcg)
from sympy import atan2

# struct-aligned param symbols (core_quadRotorParams_t fields)
c_p, cz_p, kT_p, kQ_p, Irot_p, T_max, T_min = symbols('c cz kT kQ Irot T_max T_min', positive=True)
rename = {k_T: kT_p, k_Q: kQ_p, I_rot: Irot_p, d_x: c_p, d_y: c_p, d_z: cz_p}

# reuse §2 allocation (symbolic) to go [T1..T4] -> [F, tau] and S
T1s, T2s, T3s, T4s = symbols('T1 T2 T3 T4', real=True)
M_alloc_s = M_alloc.subs(rename)
alloc = M_alloc_s * Matrix([T1s, T2s, T3s, T4s])                 # [F, tx, ty, tz]
S_ti = sum(sig[i]*sqrt([T1s,T2s,T3s,T4s][i]/kT_p) for i in range(4))

# reuse §1 state_rhs (virtual inputs), substitute the T_i allocation + struct drag
rhs_v = state_rhs.subs(rename)
rhs_ti13 = rhs_v.subs({F: alloc[0], tau_x: alloc[1], tau_y: alloc[2], tau_z: alloc[3], S_spin: S_ti})

# integrators: ref.pos - pos, and ref.yaw - yaw(q)
refx, refy, refz, ref_yaw = symbols('refx refy refz ref_yaw', real=True)
yaw_q = atan2(2*(q_w*q_z + q_x*q_y), 1 - 2*(q_y**2 + q_z**2))
ix, iy, iz, ipsi = symbols('ix iy iz ipsi', real=True)
# external perturbation forces [Fx, Fy, Fz] (world frame), applied to translational accel
user_fX, user_fY, user_fZ = symbols('user_fX user_fY user_fZ', real=True)
rhs_ti13[7] = rhs_ti13[7] + user_fX / m
rhs_ti13[8] = rhs_ti13[8] + user_fY / m
rhs_ti13[9] = rhs_ti13[9] + user_fZ / m

state_rhs_ti = Matrix([*rhs_ti13, refx - r_x, refy - r_y, refz - r_z, ref_yaw - yaw_q])

M_inv_sym = M_alloc_s.inv()   # closed-form [F,tau] -> [T1..T4]

state_syms = [r_x,r_y,r_z, q_w,q_x,q_y,q_z, v_x,v_y,v_z, w_x,w_y,w_z, ix,iy,iz,ipsi]
phys_syms  = [m, Ixx, Iyy, Izz, g, c_p, cz_p, kT_p, kQ_p, L, Irot_p, T_max, T_min]

MODEL_NAME = "QUADROTOR_FF_LQR_01"        # C++ class name (exported label)

gen = (qcg.QuadCodegen(qcg.CodegenConfig(model_name=MODEL_NAME))
       .set_state_symbols(state_syms)
       .set_input_symbols([T1s, T2s, T3s, T4s])
       .set_physics_symbols(phys_syms)
       .set_dynamics(state_rhs_ti)
       .set_lqr_gain(K)                                          # 4x16 from §4
       .set_feedforward_flat(F_ff_static, R_ref_static, omega_ff_static, tau_ff_static, M_inv_sym)
       .set_user_force_symbols([user_fX, user_fY, user_fZ]))
hpp_path, cpp_path = gen.write()
import os as _os
print("emitted:", hpp_path, cpp_path)
print(f"generated .cpp size: {_os.path.getsize(cpp_path)//1024} KB "
      f"(feedforward + dynamics share subexpressions via CSE -> fast to evaluate)")

### 6.4 Regression test — compile the C++ and compare to this notebook

Compile the generated pair standalone (`-O2 -DJUST_TESTING_DYNAMICS`, a small `test_core_defs.hpp`
mirroring the struct contract) and compare `Dynamics` and `ExecuteControl` against the Python model
on random states. Common subexpressions are shared through CSE temporaries, so the emitted math is
compact and fast; machine-precision agreement means the export is faithful.

In [ ]:
import subprocess, textwrap
from scipy.spatial.transform import Rotation

outd = gen.cfg.export_dir
# minimal test header mirroring the interface contract
open(f"{outd}/test_core_defs.hpp", "w").write(textwrap.dedent('''\
    #pragma once
    #include <array>
    typedef double core_coord_t;
    typedef std::array<core_coord_t,3> Vec3;
    typedef struct { double m,Ix,Iy,Iz,g,c,cz,kT,kQ,L,Irot,T_max,T_min; } core_quadRotorParams_t;
    typedef struct { Vec3 pos,vel,acc,jerk,snap; core_coord_t yaw,yawRate,yawAcc; } Reference_t;
    typedef struct { core_coord_t timestep,user_fX,user_fY,user_fZ; } core_stepParams_t;
'''))

# Python references (lambdified)
pv = {str(k): v for k, v in params.items()}
pv.update({'c':0.20,'cz':0.30,'T_max':36.0,'T_min':0.0,
           'kT':pv['k_T'],'kQ':pv['k_Q'],'Irot':pv['I_rot']})
def nsub(e): return e.subs({s: pv[str(s)] for s in e.free_symbols if str(s) in pv})
RFs = (refx, refy, refz, ref_yaw)
UF = [0.7, -0.4, 0.3]     # nonzero external perturbation, to exercise userF
f_dyn = lambdify((*state_syms, T1s,T2s,T3s,T4s, *RFs, user_fX, user_fY, user_fZ), nsub(state_rhs_ti), 'numpy')
AR = symbols('a_x a_y a_z j_x j_y j_z s_x s_y s_z psi psi_dot psi_ddot')
f_F  = lambdify(AR, nsub(F_ff_static), 'numpy');    f_R   = lambdify(AR, nsub(R_ref_static), 'numpy')
f_om = lambdify(AR, nsub(omega_ff_static),'numpy'); f_tau = lambdify(AR, nsub(tau_ff_static),'numpy')
Minv_n = np.array(nsub(M_inv_sym), float); Ke_n = np.array(K, float)

rng = np.random.default_rng(11); tests = []
for _ in range(5):
    q = np.r_[3,0,0,0.] + rng.normal(size=4); q /= np.linalg.norm(q)
    s = np.concatenate([rng.normal(size=3), q, rng.normal(size=3)*.5, rng.normal(size=3)*.3, rng.normal(size=4)*.2])
    u = np.abs(rng.normal(6, 1.5, size=4))
    ref = (rng.normal(size=3), rng.normal(size=3), rng.normal(size=3)*.5, rng.normal(size=3)*.3,
           rng.normal(size=3)*.2, .1*rng.normal(), .05*rng.normal(), .02*rng.normal())
    tests.append((s, u, ref))

def py_dyn(s,u,ref): p=ref[0]; return np.array(f_dyn(*s,*u,p[0],p[1],p[2],ref[5],*UF)).flatten()
def py_ec(s,ref):
    pos,vel,acc,jerk,snap,yw,yr,ya = ref; a=(*acc,*jerk,*snap,yw,yr,ya)
    Rr=np.array(f_R(*a),float).reshape(3,3); wr=np.array(f_om(*a),float).flatten(); tf=np.array(f_tau(*a),float).flatten()
    qn=s[3:7]/np.linalg.norm(s[3:7]); Rc=Rotation.from_quat([qn[1],qn[2],qn[3],qn[0]]).as_matrix()
    dth=Rotation.from_matrix(Rr.T@Rc).as_rotvec()
    cpsi,spsi=np.cos(yw),np.sin(yw)          # yaw-frame compensation (heading = ref.yaw)
    drx,dry=s[0]-pos[0],s[1]-pos[1]; dvx,dvy=s[7]-vel[0],s[8]-vel[1]; iix,iiy=s[13],s[14]
    err=np.array([cpsi*drx+spsi*dry, -spsi*drx+cpsi*dry, s[2]-pos[2], *dth,
                  cpsi*dvx+spsi*dvy, -spsi*dvx+cpsi*dvy, s[9]-vel[2],
                  s[10]-wr[0],s[11]-wr[1],s[12]-wr[2],
                  cpsi*iix+spsi*iiy, -spsi*iix+cpsi*iiy, s[15],s[16]])
    ul=-Ke_n@err; virt=np.array([float(f_F(*a))+ul[0],tf[0]+ul[1],tf[1]+ul[2],tf[2]+ul[3]])
    return np.clip(Minv_n@virt, 0.0, 36.0)

# emit driver, compile, run
def a2(x): return "{"+",".join(f"{v:.17g}" for v in x)+"}"
mod = gen.cfg.module_name
CLS = gen.cfg.model_name
drv = ['#include <cstdio>', '#include <array>', f'#include "{mod}.hpp"', 'using namespace CDS::Dynamics;',
       f"int main(){{ {CLS} m; std::array<double,3> uf{{{UF[0]},{UF[1]},{UF[2]}}}; {CLS}::StateVec s;",
       f'  {CLS}::InputVec u; Reference_t r;']
for i,(s,u,ref) in enumerate(tests):
    p,ve,ac,je,sn,yw,yr,ya = ref
    drv += [f"  s={a2(s)}; u={a2(u)};",
            f"  r.pos={a2(p)}; r.vel={a2(ve)}; r.acc={a2(ac)}; r.jerk={a2(je)}; r.snap={a2(sn)};",
            f"  r.yaw={yw:.17g}; r.yawRate={yr:.17g}; r.yawAcc={ya:.17g};",
            f'  {{auto d=m.Dynamics(s,u,r,uf); printf("D {i}"); for(auto x:d)printf(" %.17g",x); printf("\\n");}}',
            f'  {{auto c=m.ExecuteControl(s,r); printf("E {i}"); for(auto x:c)printf(" %.17g",x); printf("\\n");}}']
drv += ["  return 0; }"]
open(f"{outd}/driver.cpp","w").write("\n".join(drv))

comp = subprocess.run(["g++","-O2","-std=c++17","-DJUST_TESTING_DYNAMICS",f"-I{outd}",
                       cpp_path, f"{outd}/driver.cpp","-o",f"{outd}/driver","-lm"],
                      capture_output=True, text=True)
assert comp.returncode == 0, comp.stderr[:2000]
out = subprocess.run([f"{outd}/driver"], capture_output=True, text=True).stdout
cpp = {}
for ln in out.strip().split("\n"):
    p = ln.split(); cpp[(p[0], int(p[1]))] = np.array([float(x) for x in p[2:]])
mdyn = max(np.max(np.abs(cpp[('D',i)] - py_dyn(s,u,ref))) for i,(s,u,ref) in enumerate(tests))
mec  = max(np.max(np.abs(cpp[('E',i)] - py_ec(s,ref)))    for i,(s,u,ref) in enumerate(tests))
print(f"C++ vs Python   Dynamics: {mdyn:.2e}   ExecuteControl: {mec:.2e}")
print("PASS — generated code matches the notebook." if max(mdyn,mec) < 1e-9 else "MISMATCH")

### Notebook complete

Plant (§1) → allocation (§2) → exact flatness feedforward (§3) → LQR with four integrators (§4)
→ discussion (§5) → **verified C++ export (§6)**. The generated `Quad_FF_LQR_01.{hpp,cpp}` compiles
standalone and reproduces this notebook to machine precision, ready for the WASM runtime and the
ArduPilot-SITL bridge.
